In [1]:
# LIBRERIAS

import numpy as np
import pandas as pd
import mlflow



from scipy.stats import poisson
from sklearn.metrics import log_loss
from scipy.optimize import differential_evolution
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm
from tqdm import TqdmWarning
from IPython.display import display
from typing import Sequence, cast
import os
import hashlib
import json
import tempfile
from pathlib import Path

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# CONFIGURACIÓN DE MLFLOW (SQLite con ruta absoluta respecto al cwd del kernel)

_mlflow_db_path = (Path.cwd() / "mlflow_experimentos.db").resolve()
_mlflow_sqlite_uri = "sqlite:///" + str(_mlflow_db_path).replace("\\", "/")
mlflow.set_tracking_uri(_mlflow_sqlite_uri)
mlflow.set_experiment("Championship_Dixon_Coles_Model")
print(f"MLflow tracking DB (ruta absoluta): {_mlflow_db_path}")

MLflow tracking DB (ruta absoluta): /home/borjatortuero/Escritorio/Master/TFM/notebooks/mlflow_experimentos.db


In [3]:
# Directorio del notebook 
base_path = os.getcwd()
# Construir ruta relativa
csv_path = os.path.join(base_path, '..', 'Data', 'Segunda_inglesa_data.csv')
# Normalizar la ruta 
csv_path = os.path.abspath(csv_path)

# Cargamos el fichero
df = pd.read_csv(csv_path)

with open(csv_path, "rb") as _csv_f:
    CSV_DATA_SHA256 = hashlib.sha256(_csv_f.read()).hexdigest()

temp_date = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['Date'] = pd.to_datetime(temp_date.dt.strftime('%Y-%m-%d') + ' ' + df['Time'].astype(str))

# Ordenamos por la nueva columna Date (que ya tiene fecha y hora)
df = df.sort_values('Date').reset_index(drop=True)

# Número de partido total en la temporada para cada equipo
team_game_count = {}
home_match_nos = []
away_match_nos = []

for _, row in df.iterrows():
    h, a = row['HomeTeam'], row['AwayTeam']
    
    # Sumamos 1 al total de partidos jugados por el equipo local y el visitante
    ch = team_game_count.get(h, 0) + 1
    ca = team_game_count.get(a, 0) + 1
    
    # Guardamos qué número de partido es para cada uno en esta fila
    home_match_nos.append(ch)
    away_match_nos.append(ca)
    
    # Actualizamos el diccionario global
    team_game_count[h] = ch
    team_game_count[a] = ca

df['Home_Match_No'] = home_match_nos
df['Away_Match_No'] = away_match_nos

# Limpieza de columnas no comunes
df.dropna(inplace=True, axis=1)

df

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,Home_Match_No,Away_Match_No
0,E1,2023-08-04 20:00:00,20:00,Sheffield Weds,Southampton,1,2,A,0,1,...,1.91,1.99,1.93,2.00,1.97,2.00,1.88,1.95,1,1
1,E1,2023-08-05 15:00:00,15:00,Blackburn,West Brom,2,1,H,2,0,...,2.05,1.85,2.07,1.85,2.10,1.88,2.03,1.82,1,1
2,E1,2023-08-05 15:00:00,15:00,Bristol City,Preston,1,1,D,0,0,...,1.82,2.08,1.84,2.08,1.88,2.15,1.80,2.04,1,1
3,E1,2023-08-05 15:00:00,15:00,Middlesbrough,Millwall,0,1,A,0,0,...,1.80,2.10,1.83,2.10,1.91,2.14,1.85,1.99,1,1
4,E1,2023-08-05 15:00:00,15:00,Norwich,Hull,2,1,H,1,1,...,2.06,1.84,2.05,1.86,2.10,1.91,2.01,1.83,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099,E1,2025-05-03 12:30:00,12:30,Sheffield United,Blackburn,1,1,D,0,0,...,1.89,2.01,1.90,2.01,1.91,2.11,1.84,2.04,46,92
1100,E1,2025-05-03 12:30:00,12:30,Swansea,Oxford,3,3,D,1,1,...,2.02,1.88,2.03,1.88,2.08,1.89,2.00,1.85,92,46
1101,E1,2025-05-03 12:30:00,12:30,Norwich,Cardiff,4,2,H,3,0,...,1.86,2.04,1.88,2.03,1.96,2.06,1.86,1.99,92,92
1102,E1,2025-05-03 12:30:00,12:30,Plymouth,Leeds,1,2,A,1,0,...,1.99,1.91,1.97,1.92,1.99,1.97,1.95,1.90,92,92


In [4]:
# Nos quedamos con la primera temporada y mitad de la segunda para entrenar y con la mitad restante para predecir/validar
df_train = df.iloc[:-190].copy()
df_test = df.iloc[-190:].copy()

### **Modelo de Dixon-Coles (1997)**
El modelo de Dixon-Coles es una extensión dinámica que introduce dos mejoras críticas: un **factor de peso temporal** ($\xi$) para dar mayor relevancia a los resultados recientes y un **parámetro de dependencia** ($\rho$) que corrige la infraestimación de los marcadores bajos (0-0, 1-1, 1-0, 0-1). Este modelo utiliza una función de corrección $\tau$ para ajustar la distribución de Poisson en estos casos específicos, eliminando la asunción de independencia y logrando una estimación de la probabilidad de empate mucho más fiel a la realidad histórica del fútbol profesional.

Referencia:
Dixon, M. J., & Coles, S. G. (1997). *Modelling association football scores and inefficiencies in the football betting market*. Journal of the Royal Statistical Society: Series C (Applied Statistics).

[Acceso al artículo](https://doi.org/10.1111/1467-9876.00065)

In [5]:
def get_params_at_time_enhanced(
    target_date,
    target_home_match_no,
    target_away_match_no,
    home_team,
    away_team,
    xi,
    df_history,
    w_g,
    w_st,
    w_tot,
):
    """Histórico Date < target_date. Home_Match_No / Away_Match_No = índice acumulado de temporada del equipo
    en esa fila (todos los partidos). Decaimiento: diff = siguiente_índice_equipo - índice_fila - 1 (t=0 en el
    último partido jugado antes de target_date para ese equipo)."""
    target_date = pd.to_datetime(target_date)
    past_matches = df_history[df_history['Date'] < target_date].copy()

    if len(past_matches) == 0:
        return None, 0, 0

    q1_h = past_matches['FTHG'].quantile(0.25)
    q3_h = past_matches['FTHG'].quantile(0.75)
    iqr_h = q3_h - q1_h
    upper_fence_h = q3_h + 1.5 * iqr_h

    q1_a = past_matches['FTAG'].quantile(0.25)
    q3_a = past_matches['FTAG'].quantile(0.75)
    iqr_a = q3_a - q1_a
    upper_fence_a = q3_a + 1.5 * iqr_a

    past_matches['FTHG_lim'] = past_matches['FTHG'].clip(upper=upper_fence_h)
    past_matches['FTAG_lim'] = past_matches['FTAG'].clip(upper=upper_fence_a)

    # T_ref(T) = máximo índice acumulado de T en el pasado (como local o visitante)
    max_home_idx = past_matches.groupby('HomeTeam', sort=False)['Home_Match_No'].max()
    max_away_idx = past_matches.groupby('AwayTeam', sort=False)['Away_Match_No'].max()
    union_teams = max_home_idx.index.union(max_away_idx.index)
    t_ref = pd.concat(
        [
            max_home_idx.reindex(union_teams).fillna(0),
            max_away_idx.reindex(union_teams).fillna(0),
        ],
        axis=1,
    ).max(axis=1).astype(int)

    ht = past_matches['HomeTeam'].to_numpy()
    at = past_matches['AwayTeam'].to_numpy()

    # Siguiente índice de temporada para el local de la fila (referencia al instante target_date)
    nh = past_matches['HomeTeam'].map(t_ref).fillna(0).astype(int) + 1
    th_next = np.where(
        ht == home_team,
        target_home_match_no,
        np.where(ht == away_team, target_away_match_no, nh.to_numpy()),
    )

    na = past_matches['AwayTeam'].map(t_ref).fillna(0).astype(int) + 1
    ta_next = np.where(
        at == away_team,
        target_away_match_no,
        np.where(at == home_team, target_home_match_no, na.to_numpy()),
    )

    diff_home = np.maximum(
        0.0,
        th_next.astype(float) - past_matches['Home_Match_No'].to_numpy(dtype=float) - 1.0,
    )
    diff_away = np.maximum(
        0.0,
        ta_next.astype(float) - past_matches['Away_Match_No'].to_numpy(dtype=float) - 1.0,
    )

    past_matches['Tw_home'] = np.exp(-xi * diff_home)
    past_matches['Tw_away'] = np.exp(-xi * diff_away)

    cols_stats = ['FTHG_lim', 'HST', 'HS', 'FTAG_lim', 'AST', 'AS']
    for col in cols_stats:
        past_matches[col + '_w_h'] = past_matches[col] * past_matches['Tw_home']
        past_matches[col + '_w_a'] = past_matches[col] * past_matches['Tw_away']

    cols_wh = [c + '_w_h' for c in cols_stats]
    cols_wa = [c + '_w_a' for c in cols_stats]

    home_sums = past_matches.groupby('HomeTeam', sort=False)[cols_wh + ['Tw_home']].sum()
    away_sums = past_matches.groupby('AwayTeam', sort=False)[cols_wa + ['Tw_away']].sum()

    total_tw_h = past_matches['Tw_home'].sum()
    total_tw_a = past_matches['Tw_away'].sum()

    avg_h_g = past_matches['FTHG_lim_w_h'].sum() / total_tw_h
    avg_a_g = past_matches['FTAG_lim_w_a'].sum() / total_tw_a
    avg_h_st = past_matches['HST_w_h'].sum() / total_tw_h
    avg_a_st = past_matches['AST_w_a'].sum() / total_tw_a
    avg_h_tot = past_matches['HS_w_h'].sum() / total_tw_h
    avg_a_tot = past_matches['AS_w_a'].sum() / total_tw_a

    all_teams = home_sums.index.union(away_sums.index)

    fill_h = {'Tw_home': 0}
    fill_h.update({c: 0 for c in cols_wh})
    fill_a = {'Tw_away': 0}
    fill_a.update({c: 0 for c in cols_wa})

    home_sums = home_sums.reindex(all_teams).fillna(fill_h)
    away_sums = away_sums.reindex(all_teams).fillna(fill_a)

    stats_h = pd.DataFrame(index=all_teams)
    stats_h['FTHG'] = (home_sums['FTHG_lim_w_h'] / home_sums['Tw_home']).replace([np.inf, -np.inf, np.nan], avg_h_g)
    stats_h['HST'] = (home_sums['HST_w_h'] / home_sums['Tw_home']).replace([np.inf, -np.inf, np.nan], avg_h_st)
    stats_h['HS'] = (home_sums['HS_w_h'] / home_sums['Tw_home']).replace([np.inf, -np.inf, np.nan], avg_h_tot)
    stats_h['FTAG'] = (home_sums['FTAG_lim_w_h'] / home_sums['Tw_home']).replace([np.inf, -np.inf, np.nan], avg_a_g)
    stats_h['AST'] = (home_sums['AST_w_h'] / home_sums['Tw_home']).replace([np.inf, -np.inf, np.nan], avg_a_st)
    stats_h['AS'] = (home_sums['AS_w_h'] / home_sums['Tw_home']).replace([np.inf, -np.inf, np.nan], avg_a_tot)

    stats_a = pd.DataFrame(index=all_teams)
    stats_a['FTAG'] = (away_sums['FTAG_lim_w_a'] / away_sums['Tw_away']).replace([np.inf, -np.inf, np.nan], avg_a_g)
    stats_a['AST'] = (away_sums['AST_w_a'] / away_sums['Tw_away']).replace([np.inf, -np.inf, np.nan], avg_a_st)
    stats_a['AS'] = (away_sums['AS_w_a'] / away_sums['Tw_away']).replace([np.inf, -np.inf, np.nan], avg_a_tot)
    stats_a['FTHG'] = (away_sums['FTHG_lim_w_a'] / away_sums['Tw_away']).replace([np.inf, -np.inf, np.nan], avg_h_g)
    stats_a['HST'] = (away_sums['HST_w_a'] / away_sums['Tw_away']).replace([np.inf, -np.inf, np.nan], avg_h_st)
    stats_a['HS'] = (away_sums['HS_w_a'] / away_sums['Tw_away']).replace([np.inf, -np.inf, np.nan], avg_h_tot)

    params = pd.DataFrame(index=all_teams)
    params['Alpha_Home'] = (stats_h['FTHG']/avg_h_g)*w_g + (stats_h['HST']/avg_h_st)*w_st + (stats_h['HS']/avg_h_tot)*w_tot
    params['Beta_Home'] = (stats_h['FTAG']/avg_a_g)*w_g + (stats_h['AST']/avg_a_st)*w_st + (stats_h['AS']/avg_a_tot)*w_tot
    params['Alpha_Away'] = (stats_a['FTAG']/avg_a_g)*w_g + (stats_a['AST']/avg_a_st)*w_st + (stats_a['AS']/avg_a_tot)*w_tot
    params['Beta_Away'] = (stats_a['FTHG']/avg_h_g)*w_g + (stats_a['HST']/avg_h_st)*w_st + (stats_a['HS']/avg_h_tot)*w_tot

    return params, avg_h_g, avg_a_g

In [6]:
def predict_match_dixon_coles(row, params, avg_home_goals, avg_away_goals, rho):
    '''
    Función que predice el resultado del partido usando el modelo Dixon-Coles.
    Incorpora la dependencia entre goles (rho) para ajustar probabilidades de marcadores bajos.
    '''
    home = row['HomeTeam']   # equipo local
    away = row['AwayTeam']   # equipo visitante

    # Calcular Goles Esperados (Lambdas)
    lambda_home = params.loc[home, 'Alpha_Home'] * params.loc[away, 'Beta_Away'] * avg_home_goals
    lambda_away = params.loc[away, 'Alpha_Away'] * params.loc[home, 'Beta_Home'] * avg_away_goals

    # Generar probabilidades de marcadores
    # Calculamos hasta un número seguro de goles
    epsilon = 0.9999
    max_k_home = poisson.ppf(epsilon, lambda_home)
    max_k_away = poisson.ppf(epsilon, lambda_away)
    max_goals = int(max(max_k_home, max_k_away, 1)) + 1

    # Rango de goles
    k_values = np.arange(0, max_goals)

    # Calculamos probabilidades base (Poisson Independiente)
    prob_h = poisson.pmf(k_values, lambda_home)
    prob_a = poisson.pmf(k_values, lambda_away)

    # Matriz de probabilidades INICIAL
    # prob_matrix[i][j] es la probabilidad del marcador i-j
    prob_matrix = np.outer(prob_h, prob_a)

    # --- CORRECCIÓN DIXON-COLES ---
    # Ajustamos solo los marcadores bajos (0-0, 1-0, 0-1, 1-1) usando rho y tau

    # Definimos la función de corrección tau para los 4 casos específicos
    # Caso 0-0
    prob_matrix[0, 0] *= (1 - lambda_home * lambda_away * rho)
    # Caso 0-1
    prob_matrix[0, 1] *= (1 + lambda_home * rho)
    # Caso 1-0
    prob_matrix[1, 0] *= (1 + lambda_away * rho)
    # Caso 1-1
    prob_matrix[1, 1] *= (1 - rho)

    # Normalización de seguridad (por si rho genera valores negativos extremos raros)
    prob_matrix = np.maximum(prob_matrix, 0)
    # Re-normalizar para que la suma sea exactamente 1.0
    prob_matrix /= prob_matrix.sum()

    # Sumar probabilidades para 1X2

    # Prob Victoria Local (suma triángulo inferior)
    p_home_win = np.sum(np.tril(prob_matrix, -1))

    # Prob Empate (suma diagonal)
    p_draw = np.sum(np.diag(prob_matrix))

    # Prob Victoria Visitante (suma triángulo superior)
    p_away_win = np.sum(np.triu(prob_matrix, 1))

    return pd.Series([p_home_win, p_draw, p_away_win],
                     index=['Prob_Home', 'Prob_Draw', 'Prob_Away'])

In [7]:
# FUNCIÓN DE COSTE OPTIMIZADA (Xi, Rho)
# Utilizamos los pesos fijos obtenidos previamente en el modelo de Maher
# -------------------------------------------------------------------------
n = 0

def negative_log_likelihood_dixon_enhanced(opt_params, df_dataset, best_w_g, best_w_st, best_w_tot):
    global n
    # Desempaquetamos SOLO los 2 parámetros que vamos a optimizar
    xi_trial, rho_trial = opt_params

    # Penalizaciones lógicas
    if xi_trial < 0 or abs(rho_trial) > 1.0:
        return 9999999

    total_log_likelihood = 0
    count_valid = 0

    # Iteramos por los partidos históricos (Train)
    for idx, row in df_dataset.iterrows():
        params_t, avg_h, avg_a = get_params_at_time_enhanced(
            row['Date'],
            row['Home_Match_No'],
            row['Away_Match_No'],
            row['HomeTeam'],
            row['AwayTeam'],
            xi_trial,
            df_dataset,
            best_w_g,
            best_w_st,
            best_w_tot,
        )

        if params_t is None:
            continue

        h_team, a_team = row['HomeTeam'], row['AwayTeam']
        if h_team not in params_t.index or a_team not in params_t.index:
            continue

        # Predecimos usando la función de Dixon-Coles
        probs = predict_match_dixon_coles(row, params_t, avg_h, avg_a, rho_trial)

        p_home = probs['Prob_Home']
        p_draw = probs['Prob_Draw']
        p_away = probs['Prob_Away']

        # Calculamos probabilidad del resultado real
        if row['FTHG'] > row['FTAG']:
            prob_obs = p_home
        elif row['FTHG'] == row['FTAG']:
            prob_obs = p_draw
        else:
            prob_obs = p_away

        total_log_likelihood -= np.log(max(prob_obs, 1e-9))
        count_valid += 1

    if count_valid == 0:
        return 9999999

    # Imprimir progreso
    n += 1
    score = total_log_likelihood / count_valid

    print(f"Iter: {n} | Xi: {xi_trial:.6f} | Rho: {rho_trial:.6f} | Score: {score:.8f}")

    return score

In [8]:
# EJECUCIÓN DE LA OPTIMIZACIÓN
# -------------------------------------------------------------------------

# Buscamos la ÚLTIMA ejecución del experimento de Maher (pesos fijos para Dixon-Coles)
runs = mlflow.search_runs(
    experiment_names=["Championship_Maher_Model"],
    order_by=["start_time DESC"],
    max_results=1,
)

if runs.empty:
    raise RuntimeError(
        "No hay runs en el experimento 'Championship_Maher_Model'. "
        "Ejecuta antes el notebook Maher.ipynb hasta registrar pesos en MLflow."
    )

latest_run = runs.iloc[0]
best_w_g = float(latest_run["params.weight_goals"])
best_w_st = float(latest_run["params.weight_shots_target"])
best_w_tot = float(latest_run["params.weight_shots_total"])
maher_parent_run_id = str(latest_run["run_id"])

if mlflow.active_run() is not None:
    mlflow.end_run()

DE_SEED = 10

mlflow.start_run(run_name="Dixon_Coles_DE_xi_rho_optimization")
DIXON_RUN_ID = mlflow.active_run().info.run_id

mlflow.log_param("model_family", "dixon_coles")
mlflow.log_param("maher_parent_run_id", maher_parent_run_id)
mlflow.log_param("maher_weight_goals", best_w_g)
mlflow.log_param("maher_weight_shots_target", best_w_st)
mlflow.log_param("maher_weight_shots_total", best_w_tot)

mlflow.log_param("optimizer", "differential_evolution")
mlflow.log_param("de_strategy", "best1bin")
mlflow.log_param("de_maxiter", 1500)
mlflow.log_param("de_popsize", 20)
mlflow.log_param("de_tol", 0.001)
mlflow.log_param("de_polish", str(True))
mlflow.log_param("de_seed", DE_SEED)
mlflow.log_param("bounds_xi_rho", str(((0.0, 0.15), (-0.2, 0.2))))
mlflow.log_param("data_csv_path", csv_path)
mlflow.log_param("data_csv_sha256", CSV_DATA_SHA256)
mlflow.log_param("df_train_rows", len(df_train))

print(f"✅ Pesos de Maher importados desde MLflow (run {maher_parent_run_id})")
print("⏳ Iniciando optimización Dixon-Coles (Xi, Rho)...\n")

bounds = ((0.0, 0.15), (-0.2, 0.2))

res_dixon = differential_evolution(
    negative_log_likelihood_dixon_enhanced,
    bounds=bounds,
    args=(df_train, best_w_g, best_w_st, best_w_tot),
    strategy="best1bin",
    maxiter=1500,
    popsize=20,
    tol=0.001,
    polish=True,
    seed=DE_SEED,
)

best_xi = res_dixon.x[0]
best_rho = res_dixon.x[1]
final_objective_dixon = float(res_dixon.fun)

mlflow.log_param("xi", float(best_xi))
mlflow.log_param("rho", float(best_rho))

mlflow.log_metric("train_neg_log_likelihood", final_objective_dixon)
mlflow.log_metric("nfev", res_dixon.nfev)
mlflow.log_metric("nit", res_dixon.nit)

_payload = {
    "xi": float(best_xi),
    "rho": float(best_rho),
    "train_neg_log_likelihood": final_objective_dixon,
    "maher_weight_goals": best_w_g,
    "maher_weight_shots_target": best_w_st,
    "maher_weight_shots_total": best_w_tot,
    "maher_parent_run_id": maher_parent_run_id,
}
_td = tempfile.mkdtemp()
_json_path = os.path.join(_td, "dixon_coles_params.json")
with open(_json_path, "w", encoding="utf-8") as _jf:
    json.dump(_payload, _jf, indent=2)
mlflow.log_artifact(_json_path, artifact_path="model")

print("\n✅ ¡Optimización Dixon-Coles Completada!")
print(f"🧠 Memoria (Xi): {best_xi:.20f}")
print(f"🔗 Dependencia goles bajos (Rho): {best_rho:.20f}")
print(f"🎯 Score (Log-Likelihood por partido): {final_objective_dixon:.6f}")

✅ Pesos de Maher importados desde MLflow (run 64c675ca2f6a41229e1f894f6de3422b)
⏳ Iniciando optimización Dixon-Coles (Xi, Rho)...

Iter: 1 | Xi: 0.022515 | Rho: -0.036808 | Score: 1.05752322
Iter: 2 | Xi: 0.039535 | Rho: -0.044784 | Score: 1.05876367
Iter: 3 | Xi: 0.015634 | Rho: -0.003737 | Score: 1.05698648
Iter: 4 | Xi: 0.087360 | Rho: 0.115902 | Score: 1.06518210
Iter: 5 | Xi: 0.046657 | Rho: 0.041650 | Score: 1.05892984
Iter: 6 | Xi: 0.136181 | Rho: -0.140466 | Score: 1.07810856
Iter: 7 | Xi: 0.029297 | Rho: -0.083259 | Score: 1.05890305
Iter: 8 | Xi: 0.111848 | Rho: 0.147738 | Score: 1.07002302
Iter: 9 | Xi: 0.054939 | Rho: 0.154295 | Score: 1.06262756
Iter: 10 | Xi: 0.100579 | Rho: 0.133055 | Score: 1.06769486
Iter: 11 | Xi: 0.131400 | Rho: -0.182512 | Score: 1.07971627
Iter: 12 | Xi: 0.108704 | Rho: -0.177752 | Score: 1.07458064
Iter: 13 | Xi: 0.128740 | Rho: 0.008193 | Score: 1.07144283
Iter: 14 | Xi: 0.032707 | Rho: -0.098578 | Score: 1.05961799
Iter: 15 | Xi: 0.071426 | Rho:

In [9]:
# PREDICCIÓN ITERATIVA (ROLLING WINDOW) EN EL TEST SET

rolling_predictions = []

# Iteramos por cada partido del test set
for idx, row in df_test.iterrows():
    # Historial: Date < fecha del partido (sin fuga temporal)
    current_params, c_avg_h, c_avg_a = get_params_at_time_enhanced(
        row['Date'],
        row['Home_Match_No'],
        row['Away_Match_No'],
        row['HomeTeam'],
        row['AwayTeam'],
        best_xi,
        df,
        best_w_g,
        best_w_st,
        best_w_tot,
    )

    # Predecimos usando los parámetros actualizados a día de hoy
    pred = predict_match_dixon_coles(row, current_params, c_avg_h, c_avg_a, best_rho)

    rolling_predictions.append(pred)

# Convertimos la lista de predicciones en DataFrame
predictions_dixon = pd.DataFrame(rolling_predictions, index=df_test.index)

odds_targets = ['B365H', 'B365D', 'B365A', 'PSH', 'PSD', 'PSA']
cols_to_keep = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'] + odds_targets

# Unimos con los datos originales para evaluación
results_dixon = pd.concat([df_test[cols_to_keep], predictions_dixon], axis=1)

# Calculamos el Hit (Acierto del ganador)
outcome_conditions = [
    results_dixon['FTHG'] > results_dixon['FTAG'],
    results_dixon['FTAG'] > results_dixon['FTHG']
]
real_outcome = np.select(outcome_conditions, ['Prob_Home', 'Prob_Away'], default='Prob_Draw')
predicted_outcome = results_dixon[['Prob_Home', 'Prob_Draw', 'Prob_Away']].idxmax(axis=1)

results_dixon['Hit'] = (real_outcome == predicted_outcome).astype(int)

print("📊 Resultados de las últimas jornadas:")
cols_show = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Prob_Home', 'Prob_Draw', 'Prob_Away', 'Hit']
display(results_dixon[cols_show])

📊 Resultados de las últimas jornadas:


,Date,HomeTeam,AwayTeam,FTHG,FTAG,Prob_Home,Prob_Draw,Prob_Away,Hit
914,2025-02-05 19:45:00,Coventry,Leeds,0,2,0.277222,0.256711,0.466067,1
915,2025-02-08 12:30:00,West Brom,Sheffield Weds,2,1,0.537413,0.269743,0.192844,1
916,2025-02-08 12:30:00,Sunderland,Watford,2,2,0.505819,0.242450,0.251731,0
917,2025-02-08 15:00:00,Sheffield United,Portsmouth,2,1,0.739434,0.177586,0.082980,1
918,2025-02-08 15:00:00,Norwich,Derby,1,1,0.642366,0.203323,0.154311,0
...,...,...,...,...,...,...,...,...,...
1099,2025-05-03 12:30:00,Sheffield United,Blackburn,1,1,0.525419,0.264783,0.209798,0
1100,2025-05-03 12:30:00,Swansea,Oxford,3,3,0.527212,0.278667,0.194122,0
1101,2025-05-03 12:30:00,Norwich,Cardiff,4,2,0.544448,0.222413,0.233139,1
1102,2025-05-03 12:30:00,Plymouth,Leeds,1,2,0.173877,0.201878,0.624245,1


### **Rendimiento del modelo**

In [10]:
# --- Función para calcular RPS (Ranked Probability Score) (castiga más los errores lejanos) ---
def calculate_rps(probs, outcome_idx):
    """
    probs: lista [prob_H, prob_D, prob_A]
    outcome_idx: 0 (Home), 1 (Draw), 2 (Away)
    """
    # Acumulada Pronosticada
    cdf_p = np.cumsum(probs)   # cumsum -> el elemento n es la suma de los n anteriores
    # Acumulada Real (escalón en el resultado que ocurrió)
    cdf_o = np.zeros(3)
    cdf_o[outcome_idx:] = 1.0

    # Suma de diferencias al cuadrado / (N-1)
    return np.sum((cdf_p - cdf_o)**2) / (3-1)

In [11]:
# Función auxiliar para calcular Stake de Kelly
def get_kelly_stake(prob, odd, fraction):
    """
        ESTRATEGIA DE GESTIÓN DE CAPITAL: CRITERIO DE KELLY FRACCIONAL

    1. Kelly Stake (f*): Calcula el porcentaje óptimo de la banca para maximizar
        el crecimiento a largo plazo.
        Fórmula: f* = (Probabilidad * Cuota - 1) / (Cuota - 1)
            - El numerador representa nuestra ventaja (Edge).
            - El denominador representa el beneficio neto (Odds - 1).

    2. Kelly Fraction: Factor de seguridad que escala el stake sugerido.
            - Protege la banca contra la volatilidad y rachas negativas.
            - Absorbe el posible 'ruido' o error de estimación del modelo estadístico.
            - Ejemplo: KELLY_FRACTION = 0.05 significa que solo apostamos el 5%
              de lo que el Criterio de Kelly sugiere matemáticamente.

    3. Lógica de Decisión:
            - Si f* <= 0: El modelo no detecta ventaja real -> NO se apuesta.
            - Si f* > 0: Se apuesta proporcionalmente a la confianza y la cuota.
"""
    if odd <= 1:
        return 0
    b = odd - 1
    p = prob
    q = 1 - p
    # Fórmula Kelly: (bp - q) / b  => Equivale a: (Prob*Odd - 1) / (Odd - 1)
    f_star = (b * p - q) / b
    # Aplicamos fracción y evitamos negativos
    return max(0, f_star * fraction)

In [12]:
# --- EVALUACIÓN AVANZADA: DIXON ---

# MÉTRICAS PREDICTIVAS
# ------------------------------------------------------
rps_list_dixon = []
y_true_dixon = []
y_pred_probs_dixon = []
correct_preds_dixon = 0

for idx, row in results_dixon.iterrows():
    if row['FTHG'] > row['FTAG']:
        outcome = 0
    elif row['FTHG'] == row['FTAG']:
        outcome = 1
    else:
        outcome = 2

    probs = [row['Prob_Home'], row['Prob_Draw'], row['Prob_Away']]
    y_true_dixon.append(outcome)
    y_pred_probs_dixon.append(probs)
    rps_list_dixon.append(calculate_rps(probs, outcome))

    if np.argmax(probs) == outcome:
        correct_preds_dixon += 1

avg_rps_dixon = np.mean(rps_list_dixon)
avg_logloss_dixon = log_loss(y_true_dixon, y_pred_probs_dixon, labels=[0, 1, 2])
accuracy_dixon = correct_preds_dixon / len(results_dixon) if len(results_dixon) > 0 else 0.0

# [MLFLOW] Métricas de validación (mismo run activo abierto en la celda de optimización)
if mlflow.active_run() is None:
    raise RuntimeError(
        "No hay run MLflow activo: ejecuta antes la celda de optimización Dixon-Coles (differential evolution + MLflow)."
    )

mlflow.log_metric("validation_accuracy", accuracy_dixon)
mlflow.log_metric("validation_rps", avg_rps_dixon)
mlflow.log_metric("validation_logloss", avg_logloss_dixon)
mlflow.log_metric("validation_n_matches", len(results_dixon))

print("-" * 60)
print("📉 MÉTRICAS DE CALIDAD DEL MODELO (DIXON)")
print("-" * 60)
print(f"🎯 Accuracy:     {accuracy_dixon:.2%}")
print(f"🧩 RPS Promedio: {avg_rps_dixon:.5f}")
print(f"🔥 Log Loss:     {avg_logloss_dixon:.5f}")
print("-" * 60)

# PARÁMETROS Y PREPARACIÓN VECTORIZADA
# ------------------------------------------------------
margins_to_test = np.round(np.arange(1.01, 2.005, 0.005), 3)
kellys_to_test = np.round(np.arange(0.01, 1.005, 0.005), 3)
max_odds_to_test = np.round(np.arange(2.5, 10.1, 0.1), 1)
MIN_ODD = 1.10
unit = 1
MIN_BETS_RATIO = 0.10

total_matches_dixon = len(results_dixon)
min_bets_threshold_dixon = int(total_matches_dixon * MIN_BETS_RATIO)

# Detección dinámica de Pinnacle
has_ps_dixon = 'PSH' in results_dixon.columns
n_bookies_dixon = 2 if has_ps_dixon else 1

print(f"📊 Total Partidos: {total_matches_dixon} | Umbral Mínimo de Apuestas ({MIN_BETS_RATIO:.0%}): {min_bets_threshold_dixon}")

outcomes_dixon_arr = np.where(results_dixon['FTHG'] > results_dixon['FTAG'], 0,
                    np.where(results_dixon['FTHG'] == results_dixon['FTAG'], 1, 2))

# Aplanamiento de datos (Home, Draw, Away) x Bookies
probs_flat_dixon = np.tile(np.concatenate([results_dixon['Prob_Home'].values, results_dixon['Prob_Draw'].values, results_dixon['Prob_Away'].values]), n_bookies_dixon)

if has_ps_dixon:
    odds_flat_dixon = np.concatenate([
        results_dixon['B365H'].values, results_dixon['B365D'].values, results_dixon['B365A'].values,
        results_dixon['PSH'].values, results_dixon['PSD'].values, results_dixon['PSA'].values
    ])
    bookies_labels_dixon = np.concatenate([np.repeat('B365', total_matches_dixon * 3), np.repeat('PS', total_matches_dixon * 3)])
else:
    odds_flat_dixon = np.concatenate([results_dixon['B365H'].values, results_dixon['B365D'].values, results_dixon['B365A'].values])
    bookies_labels_dixon = np.repeat('B365', total_matches_dixon * 3)

won_flat_dixon = np.tile(np.concatenate([outcomes_dixon_arr == 0, outcomes_dixon_arr == 1, outcomes_dixon_arr == 2]), n_bookies_dixon)
fechas_flat_dixon = np.tile(np.concatenate([results_dixon['Date'].values] * 3), n_bookies_dixon)
partidos_flat_dixon = np.tile(np.concatenate([(results_dixon['HomeTeam'] + " vs " + results_dixon['AwayTeam']).values] * 3), n_bookies_dixon)
tipos_flat_dixon = np.tile(np.concatenate([np.repeat('Home', total_matches_dixon), np.repeat('Draw', total_matches_dixon), np.repeat('Away', total_matches_dixon)]), n_bookies_dixon)

# Pre-cálculo de Kelly y Profit Base
valid_mask_dixon = ~np.isnan(odds_flat_dixon) & ~np.isnan(probs_flat_dixon) & (odds_flat_dixon >= MIN_ODD)
b_flat_dixon = odds_flat_dixon - 1
with np.errstate(divide='ignore', invalid='ignore'):
    k_stake_base_dixon = (probs_flat_dixon * b_flat_dixon - (1 - probs_flat_dixon)) / b_flat_dixon
    k_stake_base_dixon = np.where(k_stake_base_dixon > 0, k_stake_base_dixon, 0)
profit_base_dixon = np.where(won_flat_dixon, k_stake_base_dixon * b_flat_dixon, -k_stake_base_dixon)

# GRID SEARCH TRIDIMENSIONAL
# ------------------------------------------------------
grid_results_dixon = []
best_roi_dixon = -float('inf')
best_history_dixon_df = pd.DataFrame()

for max_o in tqdm(max_odds_to_test, desc="🎯 Optimizando Dixon (Max Odd/Margen)"):
    mask_odd = valid_mask_dixon & (odds_flat_dixon <= max_o)

    for m in margins_to_test:
        mask_final = mask_odd & (probs_flat_dixon * odds_flat_dixon > m)
        n_bets = np.sum(mask_final)
        if n_bets < min_bets_threshold_dixon:
            continue

        profit_m = np.sum(profit_base_dixon[mask_final])
        balances = profit_m * kellys_to_test * unit
        rois = (balances / n_bets * 100)

        for i, k in enumerate(kellys_to_test):
            grid_results_dixon.append({
                'Max_Odd': max_o, 'Margen': m, 'Kelly': k,
                'Apuestas': int(n_bets), 'Balance': balances[i], 'ROI(%)': rois[i]
            })

            if rois[i] > best_roi_dixon:
                best_roi_dixon = rois[i]
                best_params_dixon = {'Margen': m, 'Kelly': k, 'Max_Odd': max_o}
                best_history_dixon_df = pd.DataFrame({
                    'Date': fechas_flat_dixon[mask_final], 'Match': partidos_flat_dixon[mask_final], 'Bookie': bookies_labels_dixon[mask_final],
                    'Type': tipos_flat_dixon[mask_final], 'Odd': odds_flat_dixon[mask_final], 'Stake': k_stake_base_dixon[mask_final] * k,
                    'Result': np.where(won_flat_dixon[mask_final], 'Win', 'Loss'), 'Ganancia': profit_base_dixon[mask_final] * k * unit
                })

# RESULTADOS Y ANÁLISIS POR RANGO
# ------------------------------------------------------
df_grid_dixon = pd.DataFrame(grid_results_dixon)
df_grid_dixon_sorted = df_grid_dixon.sort_values(by=['ROI(%)', 'Balance'], ascending=False)

print("\n" + "=" * 60)
print(f"🏆 TOP 10 CONFIGURACIONES ENCONTRADAS (DIXON) (> {min_bets_threshold_dixon} Apuestas)")
print("=" * 60)
display(df_grid_dixon_sorted.head(10).reset_index(drop=True))

if not df_grid_dixon_sorted.empty:
    best_row = df_grid_dixon_sorted.iloc[0]
    BEST_MARGIN_DIXON = best_row['Margen']
    BEST_KELLY_DIXON = best_row['Kelly']
    BEST_MAX_ODD_DIXON = best_row['Max_Odd']

    print("\n" + "*" * 60)
    print("💎 MEJOR ESTRATEGIA SELECCIONADA (DIXON)")
    print("-" * 60)
    print(f"🔹 Max Odd:  {BEST_MAX_ODD_DIXON}\n🔹 Margen:   {BEST_MARGIN_DIXON}\n🔹 Kelly:    {BEST_KELLY_DIXON}")
    print(f"🔹 ROI:      {best_row['ROI(%)']:.2f}%\n🔹 Balance:  {best_row['Balance']:.2f}")
    print(f"🔹 Apuestas: {int(best_row['Apuestas'])} ({int(best_row['Apuestas'])/(total_matches_dixon*n_bookies_dixon)*100:.1f}% de op.)")
    print("*" * 60 + "\n")

if not best_history_dixon_df.empty:
    bins, labels = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 10.0], ['1.0-1.5', '1.5-2.0', '2.0-2.5', '2.5-3.0', '3.0-4.0', '4.0+']
    best_history_dixon_df['Odd_Range'] = pd.cut(best_history_dixon_df['Odd'], bins=bins, labels=labels)
    range_analysis_dixon = best_history_dixon_df.groupby('Odd_Range', observed=False).agg({
        'Ganancia': 'sum', 'Stake': 'count', 'Result': lambda x: (x == 'Win').mean()
    }).rename(columns={'Stake': 'Número de apuestas', 'Result': 'Ratio de acierto(%)'})

    range_analysis_dixon['ROI(%)'] = (range_analysis_dixon['Ganancia'] / best_history_dixon_df.groupby('Odd_Range', observed=False)['Stake'].sum()) * 100

    print("🧐 DESGLOSE DE RENDIMIENTO POR RANGO DE CUOTA (DIXON)")
    print("-" * 60)
    display(range_analysis_dixon)

# [MLFLOW] Rejilla de apuestas: tags, artefactos y cierre del run
_bet_tmp = tempfile.mkdtemp()
mlflow.set_tag("bet_min_odd", str(MIN_ODD))
mlflow.set_tag("bet_min_bets_ratio", str(MIN_BETS_RATIO))
mlflow.set_tag("bet_margins_from_to", f"{float(margins_to_test[0])}_{float(margins_to_test[-1])}")
mlflow.set_tag("bet_kelly_from_to", f"{float(kellys_to_test[0])}_{float(kellys_to_test[-1])}")
mlflow.set_tag("bet_max_odds_from_to", f"{float(max_odds_to_test[0])}_{float(max_odds_to_test[-1])}")

if not df_grid_dixon.empty:
    _gcsv = os.path.join(_bet_tmp, "grid_results_dixon.csv")
    df_grid_dixon.to_csv(_gcsv, index=False)
    mlflow.log_artifact(_gcsv, artifact_path="betting_grid")
    mlflow.log_metric("betting_grid_n_rows", int(len(df_grid_dixon)))

if not df_grid_dixon_sorted.empty:
    _br = df_grid_dixon_sorted.iloc[0]
    mlflow.log_metric("betting_best_roi_pct", float(_br["ROI(%)"]))
    mlflow.log_metric("betting_best_balance", float(_br["Balance"]))
    mlflow.log_metric("betting_best_n_apuestas", int(_br["Apuestas"]))
    _bet_cfg = {
        "Max_Odd": float(_br["Max_Odd"]),
        "Margen": float(_br["Margen"]),
        "Kelly": float(_br["Kelly"]),
    }
    _cfg_path = os.path.join(_bet_tmp, "best_betting_config.json")
    with open(_cfg_path, "w", encoding="utf-8") as _bcf:
        json.dump(_bet_cfg, _bcf, indent=2)
    mlflow.log_artifact(_cfg_path, artifact_path="betting_grid")

mlflow.end_run()

------------------------------------------------------------
📉 MÉTRICAS DE CALIDAD DEL MODELO (DIXON)
------------------------------------------------------------
🎯 Accuracy:     51.58%
🧩 RPS Promedio: 0.21518
🔥 Log Loss:     1.02348
------------------------------------------------------------
📊 Total Partidos: 190 | Umbral Mínimo de Apuestas (10%): 19


🎯 Optimizando Dixon (Max Odd/Margen): 100%|██████████| 76/76 [00:06<00:00, 10.99it/s]



🏆 TOP 10 CONFIGURACIONES ENCONTRADAS (DIXON) (> 19 Apuestas)


,Max_Odd,Margen,Kelly,Apuestas,Balance,ROI(%)
0,9.5,1.52,1.000,19,6.563969,34.547204
1,9.6,1.52,1.000,19,6.563969,34.547204
2,9.5,1.52,0.995,19,6.531149,34.374468
3,9.6,1.52,0.995,19,6.531149,34.374468
4,9.5,1.52,0.990,19,6.498329,34.201732
5,9.6,1.52,0.990,19,6.498329,34.201732
6,9.5,1.52,0.985,19,6.465509,34.028996
7,9.6,1.52,0.985,19,6.465509,34.028996
8,9.5,1.52,0.980,19,6.432689,33.856260
9,9.6,1.52,0.980,19,6.432689,33.856260



************************************************************
💎 MEJOR ESTRATEGIA SELECCIONADA (DIXON)
------------------------------------------------------------
🔹 Max Odd:  9.5
🔹 Margen:   1.52
🔹 Kelly:    1.0
🔹 ROI:      34.55%
🔹 Balance:  6.56
🔹 Apuestas: 19 (5.0% de op.)
************************************************************

🧐 DESGLOSE DE RENDIMIENTO POR RANGO DE CUOTA (DIXON)
------------------------------------------------------------


,Ganancia,Número de apuestas,Ratio de acierto(%),ROI(%)
Odd_Range,,,,
1.0-1.5,0.000000,0,NaN,NaN
1.5-2.0,0.000000,0,NaN,NaN
2.0-2.5,-0.391363,1,0.000000,-100.000000
2.5-3.0,0.824171,1,1.000000,200.000000
3.0-4.0,1.165422,3,0.666667,147.077098
4.0+,4.965739,14,0.571429,257.070916


In [13]:
# --- VISUALIZACIÓN DE LA MEJOR ESTRATEGIA: DIXON (ROI vs BALANCE) ---

if 'best_history_dixon_df' in locals() and not df_grid_dixon_sorted.empty:
    # -------------------------------------------------------------------------
    # PREPARACIÓN DE DATOS: ESTRATEGIA MEJOR ROI
    # -------------------------------------------------------------------------
    df_viz_roi_d = best_history_dixon_df.sort_values('Date').reset_index(drop=True)
    df_viz_roi_d['Balance_Cum'] = df_viz_roi_d['Ganancia'].cumsum()
    df_viz_roi_d['MA_Balance'] = df_viz_roi_d['Balance_Cum'].rolling(window=7, min_periods=1).mean()

    # Parámetros ROI
    roi_row_d = df_grid_dixon_sorted.iloc[0]
    rd_val, rd_bal, rd_bets = roi_row_d['ROI(%)'], roi_row_d['Balance'], int(roi_row_d['Apuestas'])
    rd_marg, rd_kelly, rd_max_o = roi_row_d['Margen'], roi_row_d['Kelly'], roi_row_d['Max_Odd']

    # -------------------------------------------------------------------------
    # PREPARACIÓN DE DATOS: ESTRATEGIA MEJOR BALANCE (REGENERACIÓN VECTORIZADA)
    # -------------------------------------------------------------------------
    bal_row_d = df_grid_dixon_sorted.sort_values(by='Balance', ascending=False).iloc[0]

    if (bal_row_d['Margen'] != rd_marg) or (bal_row_d['Kelly'] != rd_kelly) or (bal_row_d['Max_Odd'] != rd_max_o):
        bd_m, bd_k, bd_mo = bal_row_d['Margen'], bal_row_d['Kelly'], bal_row_d['Max_Odd']

        mask_bal_d = valid_mask_dixon & (odds_flat_dixon <= bd_mo) & (probs_flat_dixon * odds_flat_dixon > bd_m)

        df_viz_bal_d = pd.DataFrame({
            'Date': fechas_flat_dixon[mask_bal_d],
            'Ganancia': profit_base_dixon[mask_bal_d] * bd_k * unit
        }).sort_values('Date').reset_index(drop=True)
    else:
        df_viz_bal_d = df_viz_roi_d.copy()

    df_viz_bal_d['Balance_Cum'] = df_viz_bal_d['Ganancia'].cumsum()
    df_viz_bal_d['MA_Balance'] = df_viz_bal_d['Balance_Cum'].rolling(window=7, min_periods=1).mean()

    bd_val, bd_bal_total, bd_bets = bal_row_d['ROI(%)'], bal_row_d['Balance'], int(bal_row_d['Apuestas'])
    bd_marg, bd_kelly, bd_max_o = bal_row_d['Margen'], bal_row_d['Kelly'], bal_row_d['Max_Odd']

    # -------------------------------------------------------------------------
    # DISEÑO DEL GRÁFICO (PLOTLY)
    # -------------------------------------------------------------------------
    fig = go.Figure()

    # Trazas de Balance
    fig.add_trace(go.Scatter(x=df_viz_bal_d.index, y=df_viz_bal_d['MA_Balance'], name='Tendencia (Max Balance)',
                             line=dict(color='rgba(255, 165, 0, 0.8)', width=2, dash='dot')))

    fig.add_trace(go.Scatter(x=df_viz_bal_d.index, y=df_viz_bal_d['Balance_Cum'],
                             name=f'<b>MAX BALANCE</b> ({bd_val:.2f}% ROI)',
                             line=dict(color='rgba(255, 215, 0, 0.8)', width=3)))

    # Trazas de ROI
    fig.add_trace(go.Scatter(x=df_viz_roi_d.index, y=df_viz_roi_d['MA_Balance'], name='Tendencia (Max ROI)',
                             line=dict(color='rgba(255, 255, 255, 0.5)', width=2, dash='dot')))

    fig.add_trace(go.Scatter(x=df_viz_roi_d.index, y=df_viz_roi_d['Balance_Cum'],
                             name=f'<b>MAX ROI</b> ({rd_val:.2f}% ROI)',
                             line=dict(color='rgba(255, 255, 255, 0.9)', width=3)))

    # Construcción del subtítulo con toda la información
    subtitle_text = (f"<b>Config. ROI:</b> M: {rd_marg} | K: {rd_kelly} | MaxO: {rd_max_o} | Bets: {rd_bets}  "
                     f"      /       "
                     f"<b>Config. Balance:</b> M: {bd_marg} | K: {bd_kelly} | MaxO: {bd_max_o} | Bets: {bd_bets}")

    fig.update_layout(
    template='plotly_dark',
    # Título pegado al borde superior
    title={'text': '<b>DIXON: Rendimiento Acumulado</b>', 'y': 1, 'x': 0.5, 'xanchor': 'center'},

    # Subtítulo (Anotación) con los parámetros técnicos
    annotations=[
        dict(
            text=subtitle_text,
            showarrow=False,
            xref="paper", yref="paper",
            x=0.5,
            y=1.15,
            font=dict(size=11, color="gray")
        )
    ],

    hovermode='x unified',
    xaxis=dict(title='Número de Apuesta'),
    yaxis=dict(title='Balance Acumulado (u)', zeroline=True, zerolinecolor='white'),

    # Leyenda horizontal entre el subtítulo y el gráfico
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.08,
        xanchor="center",
        x=0.5
    ),

    # Margen superior ajustado para eliminar el vacío
    margin=dict(t=100, b=50, l=50, r=50)
)

    fig.show()
else:
    print("⚠️ No hay datos suficientes para graficar. Asegúrate de ejecutar la evaluación de Dixon primero.")